<a href="https://colab.research.google.com/github/konovalyk/Python-projects/blob/main/A_B_test_statistical_significance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Iimport libraries

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest


In [2]:
from google.colab import auth
auth.authenticate_user()


In [3]:
client = bigquery.Client(project="data-analytics-mate")

### SQL query

In [4]:
query = """
WITH
 session_info AS (
 SELECT
   s.date,
   s.ga_session_id,
   sp.country,
   sp.continent,
   sp.device,
   sp.channel,
   ab.test,
   ab.test_group
 FROM
   `DA.ab_test` ab
 JOIN
   `DA.session` s
 ON
   ab.ga_session_id = s.ga_session_id
 JOIN
   `DA.session_params` sp
 ON
   sp.ga_session_id = ab.ga_session_id ),


 session_with_orders AS (
 SELECT
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group,
   COUNT(DISTINCT o.ga_session_id) AS session_with_orders
 FROM
   `DA.order` o
 JOIN
   session_info
 ON
   o.ga_session_id = session_info.ga_session_id
 GROUP BY
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group
   ),


events as (
  select
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group,
   ep.event_name,
   COUNT(ep.ga_session_id) AS event_cnt
  from
  `DA.event_params` ep
 join
 session_info
 on ep.ga_session_id = session_info.ga_session_id
group by
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group,
   ep.event_name
),
session as (
select
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group,
   COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
  from
  session_info
  group by
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group
),
account as (
select
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group,
   count(distinct acs.ga_session_id) as new_account_cnt
from
`DA.account_session` acs
join
session_info
on acs.ga_session_id = session_info.ga_session_id
group by
   session_info.date,
   session_info.country,
   session_info.continent,
   session_info.device,
   session_info.channel,
   session_info.test,
   session_info.test_group
)
select
   session_with_orders.date,
   session_with_orders.country,
   session_with_orders.continent,
   session_with_orders.device,
   session_with_orders.channel,
   session_with_orders.test,
   session_with_orders.test_group,
   'session with orders' as event_name,
   session_with_orders.session_with_orders as value
from
session_with_orders
union all
select
   events.date,
   events.country,
   events.continent,
   events.device,
   events.channel,
   events.test,
   events.test_group,
   event_name,
   events.event_cnt as value
from
events
union all
select
   session.date,
   session.country,
   session.continent,
   session.device,
   session.channel,
   session.test,
   session.test_group,
   'session' as event_name,
   session_cnt as value
from
session
union all
select
   account.date,
   account.country,
   account.continent,
   account.device,
   account.channel,
   account.test,
   account.test_group,
   'new account' as event_name,
   new_account_cnt as value
from
account;
"""

In [5]:
query_job = client.query(query)  # Executing SQL query
results = query_job.result()  # Waiting for the request to complete

In [6]:
df = results.to_dataframe()
df.to_csv('df.csv', index=False, encoding='utf-8-sig')

### Funcion

In [7]:

# Function for checking statistical significance
def analysis_ab_test(data, metrics, denominator):
    df_filtered = data[data['event_name'].isin(metrics)]
    result = []

    for test_number in df_filtered['test'].unique():
        test_data = df_filtered[df_filtered['test'] == test_number]
        group1 = test_data[test_data['test_group'] == 1]
        group2 = test_data[test_data['test_group'] == 2]

        # Checking the group size ratio
        size1 = len(group1)
        size2 = len(group2)
        ratio = min(size1, size2) / max(size1, size2) if max(size1, size2) > 0 else 0
        if ratio < 0.5:
            print(f"⚠️ Warning: Group size imbalance in test {test_number} (ratio = {ratio:.2f})")

        # Counting denominators
        denominator1 = data[(data['event_name'] == denominator) &
                            (data['test'] == test_number) &
                            (data['test_group'] == 1)]['value'].sum()
        denominator2 = data[(data['event_name'] == denominator) &
                            (data['test'] == test_number) &
                            (data['test_group'] == 2)]['value'].sum()

        for numerator_name in metrics:
            numerator1 = group1[group1['event_name'] == numerator_name]['value'].sum()
            numerator2 = group2[group2['event_name'] == numerator_name]['value'].sum()

            if denominator1 > 0 and denominator2 > 0:
                count = np.array([numerator1, numerator2])
                nobs = np.array([denominator1, denominator2])

                # Using the z-test of proportions
                z_stat, p_value = proportions_ztest(count, nobs)
                p1 = numerator1 / denominator1 * 100
                p2 = numerator2 / denominator2 * 100
                metric_change = (p2 - p1) / p1 * 100 if p1 > 0 else np.nan
                significant = p_value < 0.05
            else:
                z_stat = p_value = p1 = p2 = metric_change = np.nan
                significant = False

            result.append({
                'test_number': test_number,
                'metric': f"{numerator_name}/{denominator}",
                'numerator_count_group1': numerator1,
                'denominator_count_group1': denominator1,
                'conversion_rate_group1': p1,
                'numerator_count_group2': numerator2,
                'denominator_count_group2': denominator2,
                'conversion_rate_group2': p2,
                'metric_change': metric_change,
                'z_stat': z_stat,
                'p_value': p_value,
                'significant': significant
            })

    return result


### Metrics to check

In [8]:
# Specify metrics and denominator for verification
metrics = ['add_shipping_info', 'add_payment_info', 'new account', 'begin_checkout']
denominator = 'session'

In [9]:
# We apply the function to check the metrics in total.
result = analysis_ab_test(df, metrics, denominator)
# We convert the result to a dataframe.
result_total_df = pd.DataFrame(result)





In [10]:
# Filter data by the Organic Search channel segment
df_organic = df[df['channel'] == 'Organic Search'].reset_index(drop=True)

# We apply the function to check the metrics in segment channel Organic Search.
result_organic = analysis_ab_test(df_organic, metrics, denominator)
result_organic_df = pd.DataFrame(result_organic)


In [11]:
# Filter data by the Organic Search, mobile and Americas segments
df_organic_mobile_Americas = df[
    (df['channel'] == 'Organic Search') &
    (df['device'] == 'mobile') &
    (df['continent'] == 'Americas')
].reset_index(drop=True)


result_organic_mobile_Americas = analysis_ab_test(df_organic_mobile_Americas, metrics, denominator)
result_organic_mobile_Americas_df = pd.DataFrame(result_organic_mobile_Americas)


In [12]:


# Add a column with the segment name
result_total_df["segment_name"] = "Total"
result_organic_df["segment_name"] = "Organic"
result_organic_mobile_Americas_df["segment_name"] = "Organic_Mobile_Americas"

# We combine all df
df_all = pd.concat([result_total_df, result_organic_df, result_organic_mobile_Americas_df], ignore_index=True)

# Save to CSV
df_all.to_csv("ab_test_all_results.csv", index=False)


### [Link to Tableau Public](https://public.tableau.com/views/ABTestSignificance/significanceABTest?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)